# Transcoder conversion scripts

Convert various transcoder formats to sparsify format. Each section below is the **conversion-only** logic extracted from the model-specific notebooks (3.1 Gemma, 3.2 Llama, 3.3 Qwen).

- **Gemma**: gemma-scope transcoders from config `transcoders` list (npz or safetensors); W_enc transposed for npz; no W_skip; bfloat16.
- **Llama**: mntss `layer_*.safetensors` with W_enc, W_dec, b_enc, b_dec, **W_skip**; `skip_connection: True`.
- **Qwen**: mwhanna `layer_*.safetensors` with W_enc, W_dec, b_enc, b_dec only (no W_skip); `skip_connection: False`.

In [ ]:
"""
Convert gemma-scope transcoders (params.npz) to sparsify format.

Gemma-scope npz files contain: W_enc, W_dec, b_enc, b_dec, threshold.
  - W_enc in npz is (d_model, d_transcoder) — must be transposed to (d_transcoder, d_model)
  - W_dec is (d_transcoder, d_model) — used as-is
  - b_enc is (d_transcoder,) — used as-is
  - b_dec is (d_model,) — used as-is
  - threshold is for JumpReLU activation — not used in sparsify topk format

See: https://github.com/sg-sy/circuit-tracer (load_gemma_scope_transcoder in
circuit_tracer/transcoder/single_layer_transcoder.py)
"""

import json
import yaml
import numpy as np
import torch
from pathlib import Path
from safetensors import safe_open
from safetensors.torch import save_file
import re

HF_CACHE_ROOT = Path("/g/data/sz65/sg9944/hugging_face_cache/hub/")

def load_config(repo_path: Path) -> dict:
    """Load config from YAML or JSON file."""
    for name in ("config.yaml", "config.yml", "config.json"):
        path = repo_path / name
        if path.exists():
            with open(path, 'r') as f:
                return yaml.safe_load(f) if name.endswith(('.yaml', '.yml')) else json.load(f)
    return {}

def find_cached_file_for_hf_url(url: str, cache_root: Path) -> Path:
    """
    Resolve an hf:// URL to a local cached file (npz or safetensors).

    Example: hf://google/gemma-scope-2b-pt-transcoders/layer_0/width_16k/average_l0_76/params.npz
    -> models--google--gemma-scope-2b-pt-transcoders/snapshots/<hash>/layer_0/width_16k/average_l0_76/params.npz
    """
    m = re.match(r"hf://([^/]+)/([^/]+)/(.*)", url)
    if not m:
        raise ValueError(f"Cannot parse HF URL: {url}")
    org, repo, subpath = m.group(1), m.group(2), m.group(3)

    repo_dir = cache_root / f"models--{org}--{repo}"
    if not repo_dir.exists():
        raise FileNotFoundError(f"Repo dir not found in cache: {repo_dir}")
    snapshots_dir = repo_dir / "snapshots"
    if not snapshots_dir.exists():
        raise FileNotFoundError(f"No snapshots found in {repo_dir}")

    for snapshot in sorted(snapshots_dir.iterdir(), reverse=True):
        local_path = snapshot / subpath
        if local_path.exists():
            return local_path
        local_path_st = local_path.with_suffix('.safetensors')
        if local_path_st.exists():
            return local_path_st
    raise FileNotFoundError(f"Could not find {url} -> {subpath} under {snapshots_dir}")


def load_transcoder_weights(file_path: Path) -> dict[str, torch.Tensor]:
    """
    Load transcoder weights from either .npz or .safetensors, returning
    tensors with consistent naming and shapes for sparsify format.

    Returns dict with keys: encoder.weight, encoder.bias, W_dec, b_dec
    where encoder.weight is shape (num_latents, d_in).
    """
    suffix = file_path.suffix.lower()

    if suffix == '.npz':
        param_dict = np.load(str(file_path))
        keys = list(param_dict.keys())
        print(f"  NPZ keys: {keys}")

        # npz W_enc is (d_model, d_transcoder), transpose to (d_transcoder, d_model)
        W_enc = torch.tensor(param_dict["W_enc"]).T.contiguous()
        W_dec = torch.tensor(param_dict["W_dec"])
        b_enc = torch.tensor(param_dict["b_enc"])
        b_dec = torch.tensor(param_dict["b_dec"])

        if "threshold" in param_dict:
            threshold = param_dict["threshold"]
            print(f"  JumpReLU threshold shape: {threshold.shape} (not used in sparsify topk format)")

    elif suffix == '.safetensors':
        with safe_open(str(file_path), framework="pt", device="cpu") as f:
            keys = list(f.keys())
            print(f"  Safetensors keys: {keys}")
            W_enc = f.get_tensor("W_enc")
            W_dec = f.get_tensor("W_dec")
            b_enc = f.get_tensor("b_enc")
            b_dec = f.get_tensor("b_dec")
            # safetensors W_enc is already (d_transcoder, d_model) — no transpose needed
    else:
        raise ValueError(f"Unsupported file format: {suffix}")

    return {
        "encoder.weight": W_enc,
        "encoder.bias": b_enc,
        "W_dec": W_dec,
        "b_dec": b_dec,
    }


def convert_transcoder_to_sparsify(
    input_path: Path = Path("/g/data/sz65/sg9944/hugging_face_cache/hub/models--mntss--gemma-scope-transcoders/snapshots/9250a2d4860ce5ed5c96c14d5882b7d8162809a3/"),
    output_dir: str = "/scratch/jk87/sg9944/sparsify_mntss_transcoder_gemma22b",
):
    """
    Convert HuggingFace gemma-scope transcoder-set to sparsify format.
    Handles both params.npz (gemma-scope native) and .safetensors files.
    """
    if not input_path.exists():
        raise ValueError(f"Input path does not exist: {input_path}")
    print(f"Reading from cached repo (for config): {input_path}")
    print("=" * 80)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    hf_config = load_config(input_path)
    if not hf_config:
        raise ValueError("Unable to load HuggingFace config for transcoders!")
    print("\nHuggingFace config found:")
    print(yaml.dump(hf_config, default_flow_style=False, indent=2))

    transcoders = hf_config.get("transcoders", None)
    if not transcoders:
        raise ValueError("No transcoders list found in config!")

    print(f"\nFound {len(transcoders)} transcoder entries")
    print("=" * 80)

    converted_count = 0
    for idx, transcoder_url in enumerate(transcoders):
        try:
            layer_file = find_cached_file_for_hf_url(transcoder_url, cache_root=HF_CACHE_ROOT)
        except Exception as e:
            print(f"Could not resolve local file for {transcoder_url}: {e}")
            continue

        m = re.search(r"layer[_\-]?(\d+)", transcoder_url)
        if not m:
            print(f"Could not extract layer number from {transcoder_url}")
            continue
        layer_num = m.group(1)
        layer_name = f"layers.{layer_num}"

        print(f"\nProcessing {layer_file.name} ({layer_file.suffix}) -> {layer_name}")
        print("-" * 80)

        try:
            tensors = load_transcoder_weights(layer_file)
        except Exception as e:
            print(f"  Failed to load weights: {e}")
            continue

        num_latents, d_in = tensors["encoder.weight"].shape

        print(f"\n  Dimensions:")
        print(f"    encoder.weight: {tuple(tensors['encoder.weight'].shape)}")
        print(f"    encoder.bias:   {tuple(tensors['encoder.bias'].shape)}")
        print(f"    W_dec:          {tuple(tensors['W_dec'].shape)}")
        print(f"    b_dec:          {tuple(tensors['b_dec'].shape)}")
        print(f"    d_in={d_in}, num_latents={num_latents}, expansion={num_latents / d_in:.1f}x")

        layer_output = output_path / layer_name
        layer_output.mkdir(parents=True, exist_ok=True)

        tensors = {k: v.to(torch.bfloat16) for k, v in tensors.items()}
        save_file(tensors, str(layer_output / "sae.safetensors"))

        k = hf_config.get('k') or hf_config.get('top_k') or hf_config.get('num_active')
        if k is None:
            k = 64
            print(f"    Using default k={k}")
        else:
            print(f"    Using k={k} from config")

        cfg = {
            "d_in": int(d_in),
            "num_latents": int(num_latents),
            "expansion_factor": int(num_latents // d_in),
            "k": int(k),
            "activation": "topk",
            "transcode": True,
            "normalize_decoder": True,
            "multi_topk": False,
            "skip_connection": False,
        }
        with open(layer_output / "cfg.json", "w") as config_file:
            json.dump(cfg, config_file, indent=2)

        print(f"    Saved to: {layer_output}")
        converted_count += 1

    print("\n" + "=" * 80)
    print(f"Conversion complete! {converted_count}/{len(transcoders)} layers converted.")
    print(f"Output directory: {output_path}")
    print("=" * 80)

if __name__ == "__main__":
    convert_transcoder_to_sparsify()

---
## Llama (mntss/transcoder-Llama-3.2-1B)

Converts `layer_*.safetensors` from a cached repo. Uses **W_skip**; `skip_connection: True`.

In [ ]:
"""
Convert mntss/transcoder-Llama-3.2-1B to sparsify format.
Uses the locally cached model.
"""

import json
import yaml
from pathlib import Path
from safetensors import safe_open
from safetensors.torch import save_file

# Use your cached location
CACHED_REPO_PATH = Path("/g/data/sz65/sg9944/hugging_face_cache/hub/models--mntss--transcoder-Llama-3.2-1B/snapshots/c37a82c1ec4cea30d424850d159b17b720ce19e2/")

def load_config(repo_path: Path) -> dict:
    """Load config from YAML or JSON file."""
    config_yaml = repo_path / "config.yaml"
    config_yml = repo_path / "config.yml"
    config_json = repo_path / "config.json"

    if config_yaml.exists():
        with open(config_yaml, 'r') as f:
            return yaml.safe_load(f)
    elif config_yml.exists():
        with open(config_yml, 'r') as f:
            return yaml.safe_load(f)
    elif config_json.exists():
        with open(config_json, 'r') as f:
            return json.load(f)
    else:
        return {}

def convert_transcoder_to_sparsify(
    input_path: Path = CACHED_REPO_PATH,
    output_dir: str = "./sparsify_mntss_transcoder_llama321b",
):
    """
    Convert HuggingFace transcoder to sparsify format.

    Args:
        input_path: Path to cached HuggingFace model
        output_dir: Where to save converted files
    """

    if not input_path.exists():
        raise ValueError(f"Input path does not exist: {input_path}")

    print(f"Reading from cached repo: {input_path}")
    print("=" * 80)

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # Load config if available
    hf_config = load_config(input_path)
    if hf_config:
        print("\nHuggingFace config found:")
        print(yaml.dump(hf_config, default_flow_style=False, indent=2))

    # Find all layer safetensors files
    safetensor_files = sorted(input_path.glob("layer_*.safetensors"))

    if not safetensor_files:
        raise ValueError(f"No layer_*.safetensors files found in {input_path}")

    print(f"\nFound {len(safetensor_files)} layer files")
    print("=" * 80)

    # Process each layer
    for layer_file in safetensor_files:
        # Extract layer number from filename (e.g., "layer_0.safetensors" -> "0")
        layer_num = layer_file.stem.split('_')[1]
        layer_name = f"layers.{layer_num}"

        print(f"\n📦 Processing {layer_file.name} -> {layer_name}")
        print("-" * 80)

        with safe_open(str(layer_file), framework="pt", device="cpu") as f:
            # The keys are: W_enc, W_dec, b_enc, b_dec, W_skip
            keys = list(f.keys())
            print(f"Keys in file: {keys}")

            # Load all tensors
            W_enc = f.get_tensor("W_enc")
            W_dec = f.get_tensor("W_dec")
            b_enc = f.get_tensor("b_enc")
            b_dec = f.get_tensor("b_dec")
            W_skip = f.get_tensor("W_skip")

            # Get dimensions
            num_latents, d_in = W_enc.shape

            print(f"\nDimensions:")
            print(f"  W_enc:  {tuple(W_enc.shape)}")
            print(f"  W_dec:  {tuple(W_dec.shape)}")
            print(f"  b_enc:  {tuple(b_enc.shape)}")
            print(f"  b_dec:  {tuple(b_dec.shape)}")
            print(f"  W_skip: {tuple(W_skip.shape)}")
            print(f"\n  d_in:           {d_in}")
            print(f"  num_latents:    {num_latents}")
            print(f"  expansion:      {num_latents / d_in:.1f}x")

            # Create output directory for this layer
            layer_output = output_path / layer_name
            layer_output.mkdir(parents=True, exist_ok=True)

            # Save in sparsify format
            # Key mapping: W_enc -> encoder.weight, b_enc -> encoder.bias
            # W_dec and b_dec stay the same, W_skip stays the same
            save_file(
                {
                    "encoder.weight": W_enc,
                    "encoder.bias": b_enc,
                    "W_dec": W_dec,
                    "b_dec": b_dec,
                    "W_skip": W_skip,
                },
                str(layer_output / "sae.safetensors")
            )

            # Try to extract k from HF config if available
            k = None
            if hf_config:
                k = (hf_config.get('k') or
                     hf_config.get('top_k') or
                     hf_config.get('num_active'))

            # Fallback: Use heuristic (64x expansion suggests k ~= 32-128)
            if k is None:
                k = 64  # Conservative default for 64x expansion
                print(f"  Using default k={k}")
            else:
                print(f"  Using k={k} from config")

            # Create sparsify config
            cfg = {
                "d_in": int(d_in),
                "num_latents": int(num_latents),
                "expansion_factor": int(num_latents // d_in),
                "k": int(k),
                "activation": "topk",
                "transcode": True,  # This is a transcoder
                "normalize_decoder": True,
                "multi_topk": False,
                "skip_connection": True,  # Model has W_skip!
            }

            with open(layer_output / "cfg.json", "w") as config_file:
                json.dump(cfg, config_file, indent=2)

            print(f"\n✅ Saved to: {layer_output}")

    print("\n" + "=" * 80)
    print("✅ Conversion complete!")
    print("=" * 80)
    print(f"\nOutput directory: {output_path}")
    print(f"Converted {len(safetensor_files)} layers")

if __name__ == "__main__":
    convert_transcoder_to_sparsify()

---
## Qwen (mwhanna/qwen3-4b-transcoders)

Converts `layer_*.safetensors` from a cached repo. **No W_skip**; `skip_connection: False`.

In [ ]:
"""
Convert mwhanna/qwen3-4b-transcoders to sparsify format.
Uses the locally cached model. No W_skip (4 tensors only).
"""

import json
import yaml
from pathlib import Path
from safetensors import safe_open
from safetensors.torch import save_file

# Use your cached location
CACHED_REPO_PATH = Path("/g/data/sz65/sg9944/hugging_face_cache/hub/models--mwhanna--qwen3-4b-transcoders/snapshots/94d176260ac39ce2f882b8b09aba8c118df29bb3/")

def load_config(repo_path: Path) -> dict:
    """Load config from YAML or JSON file."""
    config_yaml = repo_path / "config.yaml"
    config_yml = repo_path / "config.yml"
    config_json = repo_path / "config.json"

    if config_yaml.exists():
        with open(config_yaml, 'r') as f:
            return yaml.safe_load(f)
    elif config_yml.exists():
        with open(config_yml, 'r') as f:
            return yaml.safe_load(f)
    elif config_json.exists():
        with open(config_json, 'r') as f:
            return json.load(f)
    else:
        return {}

def convert_transcoder_to_sparsify(
    input_path: Path = CACHED_REPO_PATH,
    output_dir: str = "/scratch/jk87/sg9944/sparsify_mwhanna_transcoder_qwen34b",
):
    """
    Convert HuggingFace transcoder to sparsify format.

    Args:
        input_path: Path to cached HuggingFace model
        output_dir: Where to save converted files
    """

    if not input_path.exists():
        raise ValueError(f"Input path does not exist: {input_path}")

    print(f"Reading from cached repo: {input_path}")
    print("=" * 80)

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # Load config if available
    hf_config = load_config(input_path)
    if hf_config:
        print("\nHuggingFace config found:")
        print(yaml.dump(hf_config, default_flow_style=False, indent=2))

    # Find all layer safetensors files
    safetensor_files = sorted(input_path.glob("layer_*.safetensors"))

    if not safetensor_files:
        raise ValueError(f"No layer_*.safetensors files found in {input_path}")

    print(f"\nFound {len(safetensor_files)} layer files")
    print("=" * 80)

    # Process each layer
    for layer_file in safetensor_files:
        # Extract layer number from filename (e.g., "layer_0.safetensors" -> "0")
        layer_num = layer_file.stem.split('_')[1]
        layer_name = f"layers.{layer_num}"

        print(f"\n📦 Processing {layer_file.name} -> {layer_name}")
        print("-" * 80)

        with safe_open(str(layer_file), framework="pt", device="cpu") as f:
            # The keys are: W_enc, W_dec, b_enc, b_dec (no W_skip)
            keys = list(f.keys())
            print(f"Keys in file: {keys}")

            # Load all tensors
            W_enc = f.get_tensor("W_enc")
            W_dec = f.get_tensor("W_dec")
            b_enc = f.get_tensor("b_enc")
            b_dec = f.get_tensor("b_dec")

            # Get dimensions
            num_latents, d_in = W_enc.shape

            print(f"\nDimensions:")
            print(f"  W_enc:  {tuple(W_enc.shape)}")
            print(f"  W_dec:  {tuple(W_dec.shape)}")
            print(f"  b_enc:  {tuple(b_enc.shape)}")
            print(f"  b_dec:  {tuple(b_dec.shape)}")
            print(f"\n  d_in:           {d_in}")
            print(f"  num_latents:    {num_latents}")
            print(f"  expansion:      {num_latents / d_in:.1f}x")

            # Create output directory for this layer
            layer_output = output_path / layer_name
            layer_output.mkdir(parents=True, exist_ok=True)

            # Save in sparsify format (no W_skip)
            save_file(
                {
                    "encoder.weight": W_enc,
                    "encoder.bias": b_enc,
                    "W_dec": W_dec,
                    "b_dec": b_dec,
                },
                str(layer_output / "sae.safetensors")
            )

            # Try to extract k from HF config if available
            k = None
            if hf_config:
                k = (hf_config.get('k') or
                     hf_config.get('top_k') or
                     hf_config.get('num_active'))

            # Fallback: Use heuristic (64x expansion suggests k ~= 32-128)
            if k is None:
                k = 64  # Conservative default for 64x expansion
                print(f"  Using default k={k}")
            else:
                print(f"  Using k={k} from config")

            # Create sparsify config
            cfg = {
                "d_in": int(d_in),
                "num_latents": int(num_latents),
                "expansion_factor": int(num_latents // d_in),
                "k": int(k),
                "activation": "topk",
                "transcode": True,  # This is a transcoder
                "normalize_decoder": True,
                "multi_topk": False,
                "skip_connection": False,  # No W_skip
            }

            with open(layer_output / "cfg.json", "w") as config_file:
                json.dump(cfg, config_file, indent=2)

            print(f"\n✅ Saved to: {layer_output}")

    print("\n" + "=" * 80)
    print("✅ Conversion complete!")
    print("=" * 80)
    print(f"\nOutput directory: {output_path}")
    print(f"Converted {len(safetensor_files)} layers")

if __name__ == "__main__":
    convert_transcoder_to_sparsify()